# MNIST MLP3: every-step TraceLogRG, all layers, 20 epochs

Third notebook for the first one-sided trace-log implementation. It runs the
correction every minibatch step for 20 epochs, with full projection strength
and no correction cap. It plots train/test accuracy, WeightWatcher `alpha`,
WeightWatcher `ERG_gap`, and the per-layer correction ratio
$\|\Delta W_{\rm correction}\|_F/\|\Delta W_{\rm AdamW}\|_F$ by epoch.
`alpha` and `ERG_gap` are read from the WeightWatcher checkpoint table; this
notebook does not fit or reconstruct either metric. No $\beta_E$ control is
introduced.

In [ ]:
from pathlib import Path
import math,sys
import matplotlib.pyplot as plt
import numpy as np,pandas as pd,torch,weightwatcher as ww
from IPython.display import display
R=None
for p in [Path.cwd(),*Path.cwd().parents]:
    if (p/"rg_trace_log").is_dir(): R=p;break
    q=p/"optimizers"/"trace_log_tracker"
    if (q/"rg_trace_log").is_dir(): R=q;break
if R is None: raise RuntimeError("Open from a clone of rg_optimizers.")
if str(R) not in sys.path: sys.path.insert(0,str(R))
from rg_trace_log import MNISTExperimentConfig,run_mnist_comparison
runs=["AdamW baseline","AdamW + TraceLogRG"]; layers=["fc1","fc2","fc3"]
rc={"AdamW baseline":"#2563A6","AdamW + TraceLogRG":"#238B57"}
bc={"fc1":"#9ECAE1","fc2":"#4292C6","fc3":"#08519C"}
gc={"fc1":"#A1D99B","fc2":"#41AB5D","fc3":"#006D2C"}
mk={"fc1":"o","fc2":"s","fc3":"^"}
cfg=MNISTExperimentConfig(
 seed=1337,epochs=20,batch_size=128,learning_rate=1e-3,weight_decay=1e-2,
 grad_clip_norm=1.0,rg_mode="one_sided",rg_normalization="weightwatcher",
 rg_gamma=.10,rg_ridge_relative=1e-6,rg_min_retained=5,
 rg_correction_scale=1.0,rg_max_correction_ratio=None,
 rg_apply_every_steps=1,rg_warmup_steps=0,ww_min_evals=10,
 ww_max_evals=None,n_log_shells=5,min_retained_for_beta=20,
 min_decades_for_beta=.50)
assert cfg.epochs==20 and cfg.rg_apply_every_steps==1
assert cfg.rg_correction_scale==1 and cfg.rg_max_correction_ratio is None
print("root",R,"torch",torch.__version__,"WeightWatcher",getattr(ww,"__version__","unknown"))
res=run_mnist_comparison(cfg,data_dir=R/"data",progress=True)
out=R/"results_every_step_all_layers_20_epochs";res.save(out)
print("saved",out.resolve())

In [ ]:
# Use WeightWatcher checkpoint output directly.
need={"run","epoch","layer_name","status","alpha","ERG_gap","trace_boundary_source"}
if need-set(res.weightwatcher.columns): raise RuntimeError("Missing WeightWatcher metrics.")
w=res.weightwatcher.copy()
w["layer"]=w.layer_name.astype(str).str.split(".").str[-1]
w=w.loc[w.status.eq("ok")&w.run.isin(runs)&w.layer.isin(layers)].copy()
w["alpha"]=pd.to_numeric(w.alpha,errors="coerce")
w["ERG_gap"]=pd.to_numeric(w.ERG_gap,errors="coerce")
if (~w.trace_boundary_source.astype(str).eq("WeightWatcher")).any():
    raise RuntimeError("WeightWatcher ERG/detX boundary missing; fallback refused.")
if w[["alpha","ERG_gap"]].isna().any().any():
    raise RuntimeError("WeightWatcher returned missing alpha or ERG_gap.")
def allplot(m,t,y,ref):
    d=w.loc[w.epoch.ge(1)]
    fig,ax=plt.subplots(figsize=(11,6),dpi=130)
    for run in runs:
        for l in layers:
            g=d.loc[d.run.eq(run)&d.layer.eq(l)].sort_values("epoch")
            col=(bc if run==runs[0] else gc)[l]
            ax.plot(g.epoch,g[m],marker=mk[l],lw=2.2,color=col,label=f"{run} — {l.upper()}")
    ax.axhline(ref,c="k",ls="--",label=f"reference = {ref:g}")
    ax.set(xlabel="Epoch",ylabel=y,title=t);ax.legend(ncol=2);plt.show()
def eachplot(m,t,y,ref):
    d=w.loc[w.epoch.ge(1)]
    for l in layers:
        fig,ax=plt.subplots(figsize=(9,5),dpi=130)
        for run in runs:
            g=d.loc[d.run.eq(run)&d.layer.eq(l)].sort_values("epoch")
            ax.plot(g.epoch,g[m],marker="o",lw=2.5,c=rc[run],label=run)
        ax.axhline(ref,c="k",ls="--",label=f"reference = {ref:g}")
        ax.set(xlabel="Epoch",ylabel=y,title=f"{t}: {l.upper()}");ax.legend();plt.show()
def acc(m,t):
    fig,ax=plt.subplots(figsize=(9,5),dpi=130)
    for run in runs:
        g=res.performance.loc[res.performance.run.eq(run)].sort_values("epoch")
        ax.plot(g.epoch,g[m],marker="o",lw=2.5,c=rc[run],label=run)
    ax.set(xlabel="Epoch",ylabel=t,title=t);ax.legend();plt.show()
print("alpha and ERG_gap source: WeightWatcher checkpoint table")
display(w[["run","epoch","layer","alpha","ERG_gap"]].head(12))

In [ ]:
allplot("alpha",r"All layers: WeightWatcher $\alpha$",r"WeightWatcher $\alpha$",2)
eachplot("alpha",r"WeightWatcher $\alpha$",r"WeightWatcher $\alpha$",2)
allplot("ERG_gap","All layers: WeightWatcher ERG gap","WeightWatcher ERG gap",0)
eachplot("ERG_gap","WeightWatcher ERG gap","WeightWatcher ERG gap",0)
acc("train_acc","Training accuracy");acc("test_acc","Test accuracy")
display(w.loc[w.epoch.ge(1)].pivot_table(index=["epoch","layer"],columns="run",
                                         values=["alpha","ERG_gap"],aggfunc="first"))

In [ ]:
# How much correction was applied to each layer in each epoch?
if res.rg_steps.empty: raise RuntimeError("No step-level correction records.")
s=res.rg_steps.copy()
s["layer"]=s.parameter.astype(str).str.replace(".weight","",regex=False).str.split(".").str[-1]
s=s.loc[s.layer.isin(layers)].copy()
for x in ["correction_ratio","base_trace_log_drift","corrected_trace_log_drift"]:
    s[x]=pd.to_numeric(s[x],errors="coerce")
s["fired"]=s.status.eq("ok");s["ratio"]=s.correction_ratio.fillna(0)
s["rf"]=s.correction_ratio.where(s.fired)
s["ab"]=s.base_trace_log_drift.abs().where(s.fired)
s["ac"]=s.corrected_trace_log_drift.abs().where(s.fired)
n=math.ceil(60000/cfg.batch_size)
b=s.groupby(["epoch","layer"],as_index=False).agg(
 steps=("global_step","nunique"),opportunities=("global_step","size"),
 fired=("fired","sum"),fired_fraction=("fired","mean"),
 failures=("status",lambda x:x.eq("geometry_failed").sum()),
 mean_ratio_all_steps=("ratio","mean"),mean_ratio_when_fired=("rf","mean"),
 max_ratio=("ratio","max"),base=("ab","sum"),corrected=("ac","sum"))
b["coverage"]=b.steps/n
b["drift_residual"]=np.where(b.base>0,b.corrected/b.base,np.nan)
display(b[["epoch","layer","opportunities","coverage","fired","fired_fraction",
           "mean_ratio_all_steps","mean_ratio_when_fired","max_ratio",
           "failures","drift_residual"]])
lab=r"$\|\Delta W_{\rm correction}\|_F/\|\Delta W_{\rm AdamW}\|_F$"
def cp(x,t,y,ylim=None):
    fig,ax=plt.subplots(figsize=(10,5),dpi=130)
    for l in layers:
        g=b.loc[b.layer.eq(l)].sort_values("epoch")
        ax.plot(g.epoch,g[x],marker=mk[l],lw=2.5,c=gc[l],label=l.upper())
    if ylim: ax.set_ylim(*ylim)
    ax.set(xlabel="Epoch",ylabel=y,title=t);ax.legend();plt.show()
cp("mean_ratio_all_steps","Mean correction applied per layer and epoch",f"Mean {lab} over all steps")
cp("mean_ratio_when_fired","Mean correction when trigger fires",f"Mean {lab} over corrected steps")
cp("max_ratio","Largest correction per layer and epoch",f"Maximum {lab}")
cp("fired_fraction","Fraction of steps corrected per layer","Corrected-step fraction",(-.02,1.02))
h=b.pivot(index="layer",columns="epoch",values="mean_ratio_all_steps").reindex(layers)
fig,ax=plt.subplots(figsize=(14,3.5),dpi=130);im=ax.imshow(h.to_numpy(float),aspect="auto",cmap="Greens")
ax.set_yticks(range(3),[x.upper() for x in layers]);ax.set_xticks(range(len(h.columns)),h.columns)
ax.set(xlabel="Epoch",ylabel="Layer",title="Mean correction / AdamW-step norm")
fig.colorbar(im,ax=ax,label=f"Mean {lab}");plt.show()

The table and green-gradient plots distinguish correction **coverage**,
**firing frequency**, mean correction over all steps, mean correction when the
trigger fires, and the maximum correction for every layer and epoch. This
remains a test of the original one-sided trace-log direction; it does not yet
add a $\beta_E$ or spectral-shape controller.